In [2]:
from sentence_transformers import CrossEncoder
import chromadb
from rank_bm25 import BM25Okapi
import numpy as np

print("All imports done!")

All imports done!


In [3]:

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Reranker ready!")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3357.03it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker ready!


In [4]:
client_db = chromadb.PersistentClient(path="../data/chromadb_rag")
collection = client_db.get_or_create_collection(name="my_pdf_docs")

all_data = collection.get()
chunks = all_data['documents']

print(f"Total chunks: {len(chunks)}")

Total chunks: 7


In [5]:
query = "What are morphological operations?"

# BM25
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)
bm25_scores = bm25.get_scores(query.lower().split())
top_bm25_indices = np.argsort(bm25_scores)[::-1][:5]
bm25_chunks = [chunks[i] for i in top_bm25_indices]

# Vector search
vector_results = collection.query(
    query_texts=[query],
    n_results=5
)
vector_chunks = vector_results['documents'][0]

# Combine
combined = list(dict.fromkeys(bm25_chunks + vector_chunks))
print(f"Combined chunks before reranking: {len(combined)}")

Combined chunks before reranking: 7


In [6]:
# Query + har chunk ka pair banao
pairs = [[query, chunk] for chunk in combined]

# Cross encoder score karo
scores = reranker.predict(pairs)

# Score ke hisaab se sort karo
ranked = sorted(zip(scores, combined), reverse=True)

print("After Reranking — Top 3 Results:")
for i, (score, chunk) in enumerate(ranked[:3]):
    print(f"\nRank {i+1} | Score: {score:.4f}")
    print(chunk[:300])
    print("---")

After Reranking — Top 3 Results:

Rank 1 | Score: 8.4218
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 
---

Rank 2 | Score: -10.0271
Fundamentals of Image Processing 1. Color Image Processing Color image processing deals with analyzing and manipulating images that contain color information. Unlike grayscale images (single intensity channel), color images typically use multiple channels such as RGB (Red, Green, Blue). Key Concepts
---

Rank 3 | Score: -10.2635
● Shape extraction ● Object detection 5. Edge Detection Techniques Edge detection identifies boundaries in an image where there is a sudden change in intensity. Common Techniques: (a) Sobel Operator ● Uses gradient approximation ● Detects horizontal and vertical edges (b) Prewitt 

In [7]:
print("=== BEFORE Reranking (Combined top result) ===")
print(combined[0][:300])

print("\n=== AFTER Reranking (Top result) ===")
print(ranked[0][1][:300])




=== BEFORE Reranking (Combined top result) ===
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 

=== AFTER Reranking (Top result) ===
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● 
